# Day 1, Session 2. Record Linkage Two Ways

You have two datasets that describe the same countries, and you want to bring a
number from one onto the other. The catch is that the names never line up. One file
calls it `Korea, Rep.` and the other says `South Korea`. This is the most common chore
in applied social science, and today you solve it two ways.

First the **character** way with Python's own `difflib`, so you see exactly how string
similarity works. Then the **semantic** way with `LinkTransformer`, a one-line tool that
matches names by meaning. Along the way you learn where each one wins and where both
hand the job back to a human.

## The two files

`countries_worldbank.csv` holds official World Bank country names. `countries_source.csv`
holds the same countries in everyday usage, plus a `gdp_growth` value we want to attach to
the official list. Both files also carry an `iso3` code. Treat `iso3` as a hidden answer key.
We only match on names, and we peek at `iso3` at the very end to grade ourselves. In real
life you would not have that key, which is the whole reason matching is hard.

In [ ]:
import pandas as pd

wb  = pd.read_csv('countries_worldbank.csv')   # iso3, country  (official)
src = pd.read_csv('countries_source.csv')       # iso3, country, gdp_growth  (everyday)

print('official names'); display(wb.head())
print('everyday names'); display(src.head())

## First try. A plain merge

The natural move is a pandas merge on the country name. Watch how many of the 43 rows
actually join.

In [ ]:
naive = wb.merge(src[['country', 'gdp_growth']], on='country', how='inner')
print(f'{len(naive)} of {len(wb)} countries joined on an exact name match')
naive[['country', 'gdp_growth']]

Almost nothing joined. Only the handful of countries whose names happen to be spelled
identically in both files came through. Every `Korea, Rep.` against `South Korea` was
dropped on the floor. We need matching that tolerates the differences.

## Approach A. Character matching with difflib

`difflib.SequenceMatcher` scores how similar two strings are on a 0 to 1 scale, counting
the characters they share in order. We score each everyday name against every official
name and keep the closest one.

In [ ]:
import difflib

official = wb['country'].tolist()

def best_match(name, choices):
    scored = [(difflib.SequenceMatcher(None, name.lower(), c.lower()).ratio(), c) for c in choices]
    score, match = max(scored)          # max compares the score first
    return match, round(score, 2)

src['match'], src['score'] = zip(*[best_match(n, official) for n in src['country']])
src[['country', 'match', 'score']].head(15)

### A threshold turns scores into a decision

A high score is a confident match. A low score means difflib is guessing. We accept
matches at or above a threshold and send the rest to human review. Our pilot found 0.8 is
a sensible cutoff for this data.

In [ ]:
THRESHOLD = 0.8
src['accepted'] = src['match'].where(src['score'] >= THRESHOLD)

auto   = src['accepted'].notna().sum()
review = src['accepted'].isna().sum()
print(f'auto-accepted {auto}, sent to review {review}')
src.loc[src['accepted'].isna(), ['country', 'match', 'score']]

### Grade the character matcher

Now we unlock the hidden `iso3` key and check how many top guesses were actually right.

In [ ]:
off_iso = dict(zip(wb['country'], wb['iso3']))
src['guess_iso3'] = src['match'].map(off_iso)
src['correct'] = src['guess_iso3'] == src['iso3']

print(f"difflib top-1 correct on {src['correct'].sum()} of {len(src)}")
src.loc[~src['correct'], ['country', 'match', 'score']]

Look at what difflib got wrong. `USA`, `UK`, `UAE`, `Burma`, `Palestine`. These are
acronyms and renames. They share almost no letters with the official name, so a character
score cannot see the link. No amount of tuning fixes this, because the similarity is in the
**meaning**, not the spelling. That is the opening for the second approach.

## Approach B. Semantic matching with embeddings

Character matching cannot see that `Burma` and `Myanmar` are the same place, since they
share no letters. Semantic matching can. It turns each name into a vector with a language
model, then links the names whose vectors sit closest. We use `sentence-transformers`, which
runs on Colab with no fragile setup.

The first run downloads a small model, which takes a moment on the free Colab CPU.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
E_off = model.encode(official, normalize_embeddings=True)
E_src = model.encode(src['country'].tolist(), normalize_embeddings=True)

sims = E_src @ E_off.T                    # cosine similarity, both sides normalized
src['semantic_match'] = [official[i] for i in sims.argmax(axis=1)]
src['semantic_iso3'] = src['semantic_match'].map(off_iso)

hits = (src['semantic_iso3'] == src['iso3']).sum()
print(f'semantic top-1 correct on {hits} of {len(src)}')
src[['country', 'match', 'semantic_match']]

Check the rows difflib missed. `Burma` now links to `Myanmar`, `UK` to `United Kingdom`,
`UAE` to `United Arab Emirates`. The model learned during pretraining that these names
refer to the same place, even though they look nothing alike.

A package called `LinkTransformer` wraps this same idea in a single `lt.merge` call. We use
the few-line version here so the mechanism stays visible and the notebook runs with no extra
setup.

## What each method is good for

Neither tool is the winner. They are good at different things.

- **difflib** wins on formatting noise. Punctuation, a dropped `The`, `Republic` against
  `Rep.`, a typo. It is stdlib, instant, and transparent.
- **Semantic matching** wins on meaning. Acronyms, translations, and renames like
  `Burma` and `Myanmar` that share no letters.
- **Both** stumble on genuine ambiguity. Two Congos and two Koreas look alike and mean
  almost the same thing, so a person still has to decide. Good pipelines auto-accept the
  confident matches and route the rest to a human.

You also just used a language model as a black box that somehow knew `Burma` is `Myanmar`.
How does it turn a name into a vector of meaning? That is exactly what we open up on Day 2.